In [ ]:
'''
**Authors:** Davide Carecci  
**Created:** February 2024  
Script to plot and visualize the outputs of UI_selectorPI_operative.py (UIT actual closed-loop responses).
'''
import os
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from general_utils.old_uit_functions import read_csv_files

In [ ]:
# In[1]: READ CSV FILES GENERATED BY UIT_SELECTORPI_OPERATIVE.PY
strings_list = ['Output_selector', 'Boundary_calc_conditions','Header','Follower']
result = read_csv_files(strings_list, 
                        directory=r"C:\Users\lenovo\OneDrive - Politecnico di Milano\Work_cloud\DOTTORATO\Sperimentazione UIT\SelectorPI controller\UIT implementation\Output")

In [ ]:
# In[2]: COMPUTATION OF DOSED TOMATO PER DAY
pump_dose_per_minute = 10 # mL/minute
on_minutes_nominal = 1 # Nominal control action load (in min/control interval)
dt = 2.5 # Control interval (in hours)
df = result['Output_selector']
df['Timestamp_control'] = pd.to_datetime(df['Timestamp_control'])

# Define a function to perform the operation for each day
def process_day(day_df):
    # Multiply final_control_signal by 1e6 to convert to mL/day, then by 2.5/24 to get per 2.5 hours, and add the nominal/initial dosage
    day_sum = (day_df['final_control_signal']*1e6*dt/24 + pump_dose_per_minute*on_minutes_nominal).sum()
    return pd.DataFrame({'date': [day_df['Timestamp_control'].iloc[0].date()], 'day_sum': [day_sum]})

# Group by day and calculate the sum for each group
on_minutes_tot_daily_sum = df.groupby(df['Timestamp_control'].dt.date)['on_minutes_tot'].sum().reset_index()
on_minutes_tot_daily_sum['on_minutes_tot'] = on_minutes_tot_daily_sum['on_minutes_tot']*pump_dose_per_minute # (mL/day)

# Apply the function for each day in the original DataFrame
final_control_signal_daily_sum = df.groupby(df['Timestamp_control'].dt.date).apply(process_day).reset_index(drop=True)
# Concatenate the two DataFrames
result_combined = pd.concat([on_minutes_tot_daily_sum, final_control_signal_daily_sum], axis=1)
print(result_combined)

In [ ]:
# In[3]: PLOT CH4 MEASURE AND SETPOINT OVERALL TIME SERIES
df1 = result['Boundary_calc_conditions']
df2 = result['Output_selector']
df1['Timestamp_control'] = pd.to_datetime(df1['Timestamp_control'])
df2['Timestamp_control'] = pd.to_datetime(df2['Timestamp_control'])
# Plot the overall time series
plt.figure(figsize=(12, 6))
plt.plot(df1['Timestamp_control'], df1['ch4_measure'], color='r', label='ch4_measure',linewidth=0.5,marker='o')
plt.plot(df1['Timestamp_control'], df1['ch4_setpoint'], color='b', label='ch4_setpoint',linewidth=0.5,marker='o')
plt.xlabel('Timestamp')
plt.ylabel('Variable Value')
plt.title('Overall Time Series')
plt.legend()
plt.xticks(rotation=90)
plt.tight_layout()
# Set x-axis limits with timestamps (from 10:30 AM to 12:30 PM)
start_time = '2024-01-31 00:00:00'
end_time = '2024-02-03 00:00:00'
plt.xlim(mdates.date2num(np.datetime64(start_time)), mdates.date2num(np.datetime64(end_time)))
ax = plt.gca()
#ax.set_ylim([42, 42.4])
#ax.set_xlim([15000,15015])
# Set x-axis formatting to display dates as 'YYYY-MM-DD'
date_format = mdates.DateFormatter('%Y-%m-%d %H') #('%Y-%m-%d %H:%m')
ax.xaxis.set_major_formatter(date_format)
# Set the locator for the x-axis to show every day
ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
plt.grid()
plt.show()

In [ ]:
# In[4]: PLOT CH4 MEASURE AND SETPOINT OVERALL TIME SERIES WITH CONTROL SIGNAL AND SELECTION STATUS MASK
df1 = result['Boundary_calc_conditions']
df2 = result['Output_selector']
df1['Timestamp_control'] = pd.to_datetime(df1['Timestamp_control'])
df2['Timestamp_control'] = pd.to_datetime(df2['Timestamp_control'])
# Plot the overall time series
fig, ax1 = plt.subplots(figsize=(16, 6))
ax1.plot(df1['Timestamp_control'], df1['ch4_measure'], color='r', label='Qch4_measure',linewidth=0.5,marker='o')
ax1.plot(df1['Timestamp_control'], df1['ch4_setpoint'], color='b', label='Qch4_setpoint',linewidth=0.5,marker='o')
ax1.set_xlabel('Timestamp')
ax1.set_ylabel('Qch4 [L/h]')
ax1.legend(loc='upper left')
#-----------------------------------------------------------------------------------------------------
# Create a second y-axis sharing the same x-axis
ax2 = ax1.twinx()
ax2.plot(df2['Timestamp_control'], df2['final_control_signal']*1e6, color='g', label='final_control_signal',linewidth=0.5,marker='o')
ax2.set_ylabel('u [mL/day]')
ax2.legend(loc='upper right')
#-----------------------------------------------------------------------------------------------------
# Add vertical dotted lines at specific time instants
specific_time_instants = ['2024-1-31 10:30','2024-2-2 10:30',
                          '2024-2-5 10:30','2024-2-7 15:00'
                         ]
for time_instant in specific_time_instants:
    ax1.axvline(pd.to_datetime(time_instant), color='grey', linestyle='dashed')#, label=f'{time_instant}')
#----------------------------------------------------------------------------------------------------
# Set x-axis limits with timestamps (from 10:30 AM to 12:30 PM)
start_time = '2024-01-31 12:00:00'
end_time = '2024-02-12 00:00:00'
ax2.set_xlim(mdates.date2num(np.datetime64(start_time)), mdates.date2num(np.datetime64(end_time)))
#ax.set_ylim([42, 42.4])
#ax.set_xlim([15000,15015])
# Set x-axis formatting to display dates as 'YYYY-MM-DD'
date_format = mdates.DateFormatter('%Y-%m-%d %H') #('%Y-%m-%d %H:%m')
ax2.xaxis.set_major_formatter(date_format)
# Set the locator for the x-axis to show every day
ax2.xaxis.set_major_locator(mdates.HourLocator(interval=3))
# Fill the area where the specified column contains the specified value with grey color
mask = df2['active'] == 'Header'
ax1.fill_between(df2['Timestamp_control'], ax1.get_ylim()[0], ax1.get_ylim()[1], where=mask, color='grey', alpha=0.3)
plt.title('Methane flowrate tracking')
ax1.tick_params(rotation=90)
plt.tight_layout()
ax1.grid()
plt.show()

In [ ]:
# In[5]: PLOT CO2/CH4 MEASURE AND SETPOINT OVERALL TIME SERIES WITH CONTROL SIGNAL AND SELECTION STATUS MASK
df1 = result['Boundary_calc_conditions']
df2 = result['Output_selector']
df1['Timestamp_control'] = pd.to_datetime(df1['Timestamp_control'])
df2['Timestamp_control'] = pd.to_datetime(df2['Timestamp_control'])
# Plot the overall time series
fig, ax1 = plt.subplots(figsize=(16, 6))
ax1.plot(df1['Timestamp_control'], df1['co2/ch4_measure'], color='r', label='co2/ch4_measure',linewidth=0.5,marker='o')
ax1.plot(df1['Timestamp_control'], df1['co2/ch4_setpoint'], color='b', label='co2/ch4_setpoint',linewidth=0.5,marker='o')
ax1.set_xlabel('Timestamp')
ax1.set_ylabel('co2/ch4')
ax1.legend(loc='upper left')
#-----------------------------------------------------------------------------------------------------
# Create a second y-axis sharing the same x-axis
ax2 = ax1.twinx()
ax2.plot(df2['Timestamp_control'], df2['final_control_signal']*1e6, color='g', label='final_control_signal',linewidth=0.5,marker='o')
ax2.set_ylabel('u [mL/day]')
ax2.legend(loc='upper right')
#-----------------------------------------------------------------------------------------------------
# Add vertical dotted lines at specific time instants
specific_time_instants = ['2024-1-31 10:30','2024-2-2 10:30',
                          '2024-2-5 10:30','2024-2-7 15:00','2024-2-9 16:15'
                         ]
for time_instant in specific_time_instants:
    ax1.axvline(pd.to_datetime(time_instant), color='grey', linestyle='dashed')#, label=f'{time_instant}')
#----------------------------------------------------------------------------------------------------
# Set x-axis limits with timestamps (from 10:30 AM to 12:30 PM)
start_time = '2024-01-31 12:00:00'
end_time = '2024-02-12 00:00:00'
ax2.set_xlim(mdates.date2num(np.datetime64(start_time)), mdates.date2num(np.datetime64(end_time)))
#ax.set_ylim([42, 42.4])
#ax.set_xlim([15000,15015])
# Set x-axis formatting to display dates as 'YYYY-MM-DD'
date_format = mdates.DateFormatter('%Y-%m-%d %H') #('%Y-%m-%d %H:%m')
ax2.xaxis.set_major_formatter(date_format)
# Set the locator for the x-axis to show every day
ax2.xaxis.set_major_locator(mdates.HourLocator(interval=3))
ax1.axhline(0.814834862+0.05, color='grey', linestyle='dashed', label='uHigh')
ax1.axhline(0.814834862+0.1, color='grey', linestyle='dashed', label='uLow')
# Fill the area where the specified column contains the specified value with grey color
mask = df2['active'] == 'Header'
ax1.fill_between(df2['Timestamp_control'], ax1.get_ylim()[0], ax1.get_ylim()[1], where=mask, color='grey', alpha=0.3)
plt.title('co2/ch4 gas ratio "tracking"')
ax1.tick_params(rotation=90)
plt.tight_layout()
ax1.grid()
plt.show()

In [ ]:
# In[5]: PLOT HEADER AND FOLLOWER PROPORTIONAL AND INTEGRAL ACTIONS OVERALL TIME SERIES WITH SELECTION STATUS MASK
df1 = result['Header']
df2 = result['Follower']
df1['timestamp'] = pd.to_datetime(df1['timestamp'])
df2['timestamp'] = pd.to_datetime(df2['timestamp'])
# Plot the overall time series
fig, ax1 = plt.subplots(figsize=(16, 6))
ax1.plot(df1['timestamp'], df1['u_p']*1e6, color='r', label='Header up',linewidth=0.5,marker='o')
ax1.plot(df2['timestamp'], df2['u_p']*1e6, color='b', label='Follower up',linewidth=0.5,marker='o')
ax1.plot(df1['timestamp'], df1['u_i']*1e6, color='orange', label='Header ui',linewidth=0.5,marker='o')
ax1.plot(df2['timestamp'], df2['u_i']*1e6, color='lightblue', label='Follower ui',linewidth=0.5,marker='o')
ax1.set_xlabel('Timestamp')
ax1.set_ylabel('u [mL/day]')
ax1.legend(loc='upper left')
#-----------------------------------------------------------------------------------------------------
# Create a second y-axis sharing the same x-axis
ax2 = ax1.twinx()
#ax2.plot(df2['Timestamp_control'], df2['final_control_signal']*1e6, color='g', label='final_control_signal',linewidth=0.5,marker='o')
#ax2.set_ylabel('u')
#ax2.legend(loc='upper right')
#-----------------------------------------------------------------------------------------------------
# Add vertical dotted lines at specific time instants
specific_time_instants = ['2024-1-31 10:30','2024-2-2 10:30',
                          '2024-2-5 10:30'
                         ]
for time_instant in specific_time_instants:
    ax1.axvline(pd.to_datetime(time_instant), color='grey', linestyle='dashed')#, label=f'{time_instant}')
#----------------------------------------------------------------------------------------------------
# Set x-axis limits with timestamps (from 10:30 AM to 12:30 PM)
start_time = '2024-01-31 12:00:00'
end_time = '2024-02-09 00:00:00'
ax2.set_xlim(mdates.date2num(np.datetime64(start_time)), mdates.date2num(np.datetime64(end_time)))
#ax.set_ylim([42, 42.4])
#ax.set_xlim([15000,15015])
# Set x-axis formatting to display dates as 'YYYY-MM-DD'
date_format = mdates.DateFormatter('%Y-%m-%d %H') #('%Y-%m-%d %H:%m')
ax2.xaxis.set_major_formatter(date_format)
# Set the locator for the x-axis to show every day
ax2.xaxis.set_major_locator(mdates.HourLocator(interval=3))
# Fill the area where the specified column contains the specified value with grey color
mask = result['Output_selector']['active'] == 'Header'
ax1.fill_between(result['Output_selector']['Timestamp_control'], ax1.get_ylim()[0], ax1.get_ylim()[1], where=mask, color='grey', alpha=0.3)
plt.title('Proportional and integral actions')
ax1.tick_params(rotation=90)
plt.tight_layout()
ax1.grid()
plt.show()